In [11]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

In [12]:
spark = (
    SparkSession
    .builder
    .appName("zoomcamp")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC") 
    .getOrCreate()
)

In [13]:
!java --version
pyspark.__file__

openjdk 17.0.18 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)


'/workspaces/data-zoomcamp/module6/.venv/lib/python3.12/site-packages/pyspark/__init__.py'

-------------------------
**Question 1** Question 1: Install Spark and PySpark


In [14]:
spark.version

'4.1.1'

-------------------------------------

**Question 2** Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

In [15]:
df_homework = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [16]:
df_homework.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [17]:
df_part = df_homework.repartition(4)


In [19]:
df_part.write.parquet('data/pq/yellow/', mode='overwrite')

In [20]:
!tree -h --du 

[464M]  .
├── [ 16K]  04_pyspark.ipynb
├── [  45]  README.md
├── [101M]  data
│   └── [101M]  pq
│       └── [101M]  yellow
│           ├── [   0]  _SUCCESS
│           ├── [ 25M]  part-00000-5bdd4ed7-7e37-45fb-9610-bd1a5ca3bb7c-c000.snappy.parquet
│           ├── [ 25M]  part-00001-5bdd4ed7-7e37-45fb-9610-bd1a5ca3bb7c-c000.snappy.parquet
│           ├── [ 25M]  part-00002-5bdd4ed7-7e37-45fb-9610-bd1a5ca3bb7c-c000.snappy.parquet
│           └── [ 25M]  part-00003-5bdd4ed7-7e37-45fb-9610-bd1a5ca3bb7c-c000.snappy.parquet
├── [295M]  fhvhv_tripdata_2021-01.parquet
├── [  85]  main.py
├── [ 21K]  mod6_homework.ipynb
├── [ 229]  pyproject.toml
├── [ 239]  test_spark.py
├── [189K]  uv.lock
└── [ 68M]  yellow_tripdata_2025-11.parquet

 768M used in 4 directories, 14 files


------------------------

**Question 3** How many taxi trips were there on the 15th of November?

In [21]:
df_part.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [22]:
df_part = df_part \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [23]:
df_part \
    .withColumn('pickup_date', F.to_date(df_part.pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

In [24]:
df_part.createOrReplaceTempView ('yellow_2025_11')

In [25]:
spark.sql("""
SELECT COUNT(*) 
FROM yellow_2025_11
WHERE to_date(pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [26]:
df_part.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

------------------------------------------

**Question 4** What is the length of the longest trip in the dataset in hours?

Notes: difference in hours between the pickup_datetime and dropoff_datetime

And then go from seconds to hours : 60 seconds * 60 minutes = 



In [27]:
df_part.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [28]:
spark.conf.get("spark.sql.session.timeZone")

'UTC'

In [42]:
# If they are currently strings, convert them first
df_part = df_part.withColumn('pickup_datetime', F.to_timestamp('pickup_datetime')) \
       .withColumn('dropoff_datetime', F.to_timestamp('dropoff_datetime'))

In [43]:
df_part = df_part \
    .withColumn('duration_seconds', F.col('dropoff_datetime').cast('long') - F.col('pickup_datetime').cast('long'))

In [49]:
df_part = df_part.withColumn('pickup_unix', F.unix_seconds(F.col('pickup_datetime'))) \
                .withColumn('dropoff_unix', F.unix_seconds(F.col('dropoff_datetime'))) \
                .withColumn('pickup_date', F.to_date(df_part.pickup_datetime))

# Calculate duration
df_part = df_part.withColumn('duration_unix', F.col('dropoff_unix') - F.col('pickup_unix'))

In [51]:
df_part \
    .select('pickup_datetime', 'dropoff_datetime', 'trip_distance', 'duration_seconds', 'duration_unix', 'pickup_unix', 'dropoff_unix') \
    .limit(5) \
    .show()

[Stage 38:>                                                         (0 + 2) / 2]

+-------------------+-------------------+-------------+----------------+-------------+-----------+------------+
|    pickup_datetime|   dropoff_datetime|trip_distance|duration_seconds|duration_unix|pickup_unix|dropoff_unix|
+-------------------+-------------------+-------------+----------------+-------------+-----------+------------+
|2025-11-18 10:29:35|2025-11-18 11:01:18|         9.48|            1903|         1903| 1763461775|  1763463678|
|2025-11-12 09:23:59|2025-11-12 09:40:01|         1.64|             962|          962| 1762939439|  1762940401|
|2025-11-15 21:52:54|2025-11-15 22:22:46|         4.03|            1792|         1792| 1763243574|  1763245366|
|2025-11-09 10:19:41|2025-11-09 10:28:48|          1.9|             547|          547| 1762683581|  1762684128|
|2025-11-04 20:44:29|2025-11-04 21:02:26|         3.63|            1077|         1077| 1762289069|  1762290146|
+-------------------+-------------------+-------------+----------------+-------------+-----------+------

In [33]:
df_part \
    .withColumn('duration_hour', F.col('duration_seconds')/3600) \
    .groupBy('pickup_date') \
    .max('duration_hour') \
    .orderBy('max(duration_hour)', ascending=False) \
    .limit(5) \
    .show()

[Stage 26:===========================================>              (3 + 1) / 4]

+-----------+------------------+
|pickup_date|max(duration_hour)|
+-----------+------------------+
| 2025-11-26| 90.64666666666666|
| 2025-11-27| 76.94833333333334|
| 2025-11-03| 76.21388888888889|
| 2025-11-07| 69.28861111111111|
| 2025-11-18| 67.08055555555555|
+-----------+------------------+



In [55]:
df_part.createOrReplaceTempView ('yellow_2025_11')

spark.sql("""
SELECT
    to_date(pickup_datetime) AS pickup_date,
    MAX(duration_unix/ 3600) AS duration
FROM 
    yellow_2025_11
GROUP BY
    1
ORDER BY
    2 DESC
LIMIT 10;
""").show()

[Stage 55:=============================>                            (2 + 2) / 4]

+-----------+------------------+
|pickup_date|          duration|
+-----------+------------------+
| 2025-11-26| 90.64666666666666|
| 2025-11-27| 76.94833333333334|
| 2025-11-03| 76.21388888888889|
| 2025-11-07| 69.28861111111111|
| 2025-11-18| 67.08055555555555|
| 2025-11-22| 63.36833333333333|
| 2025-11-01|56.382222222222225|
| 2025-11-05|42.720555555555556|
| 2025-11-06|41.614444444444445|
| 2025-11-24|38.074444444444445|
+-----------+------------------+



---------------------------------

**Question 6**: Least frequent pickup location zone

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [34]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv


--2026-03-10 02:56:31--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.97, 52.85.39.65, 52.85.39.117, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.97|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 02:56:32 (1.31 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [36]:
df_zones = spark.read.csv('taxi_zone_lookup.csv')

In [37]:
df_zones.show()

+----------+-------------+--------------------+------------+
|       _c0|          _c1|                 _c2|         _c3|
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhatta

In [62]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

In [63]:
df_zones.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



**GO** - what is the name of the LEAST frequent pickup location Zone

In [56]:
df_part.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- duration_seconds: long (nullable = true)
 |-- pickup_unix: long (nullable = true)
 |-- dr

In [60]:
df_result = df_part \
    .groupBy("PULocationID") \
    .count() \
    .orderBy(F.asc("count"))

df_result.show()

[Stage 73:=============================>                            (2 + 2) / 4]

+------------+-----+
|PULocationID|count|
+------------+-----+
|          84|    1|
|         105|    1|
|           5|    1|
|         187|    3|
|         111|    4|
|         109|    4|
|         204|    4|
|         199|    4|
|           2|    5|
|         251|   12|
|          59|   14|
|         176|   14|
|         245|   14|
|         172|   14|
|         253|   15|
|          27|   16|
|         206|   17|
|          30|   18|
|         156|   21|
|         118|   22|
+------------+-----+
only showing top 20 rows


In [61]:
df_part.createOrReplaceTempView("trips")

spark.sql("""
    SELECT 
        PULocationID, 
        COUNT(*) as trip_count
    FROM 
        trips
    GROUP BY 
        PULocationID
    ORDER BY 
        trip_count ASC
    LIMIT 10
""").show()

[Stage 79:=============================>                            (2 + 2) / 4]

+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|          84|         1|
|         105|         1|
|           5|         1|
|         187|         3|
|         111|         4|
|         109|         4|
|         204|         4|
|         199|         4|
|           2|         5|
|         251|        12|
+------------+----------+



In [64]:
# Step 1: Get the counts
df_counts = df_part.groupBy("PULocationID").count()

# Step 2: Join with the Zones table
# We join 'PULocationID' from our counts to 'LocationID' in the lookup table
df_result = df_counts.join(df_zones, df_counts.PULocationID == df_zones.LocationID, how='left')

# Step 3: Sort by the busiest zones
df_result.select("Zone", "Borough", "count") \
    .orderBy(F.asc("count")) \
    .show(10)

[Stage 87:===========================================>              (3 + 1) / 4]

+--------------------+-------------+-----+
|                Zone|      Borough|count|
+--------------------+-------------+-----+
|Eltingville/Annad...|Staten Island|    1|
|Governor's Island...|    Manhattan|    1|
|       Arden Heights|Staten Island|    1|
|       Port Richmond|Staten Island|    3|
| Green-Wood Cemetery|     Brooklyn|    4|
|   Rossville/Woodrow|Staten Island|    4|
|         Great Kills|Staten Island|    4|
|       Rikers Island|        Bronx|    4|
|         Jamaica Bay|       Queens|    5|
|         Westerleigh|Staten Island|   12|
+--------------------+-------------+-----+
only showing top 10 rows


In [65]:
df_part.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

spark.sql("""
    SELECT 
        PULocationID, 
        Borough,
        Zone,
        service_zone,
        COUNT(*) as trip_count
    FROM 
        trips
         INNER JOIN zones ON trips.PULocationID = zones.LocationID
    GROUP BY 
        PULocationID, 
        Borough,
        Zone,
        service_zone
    ORDER BY 
        trip_count ASC
    LIMIT 10
""").show()

[Stage 94:=============================>                            (2 + 2) / 4]

+------------+-------------+--------------------+------------+----------+
|PULocationID|      Borough|                Zone|service_zone|trip_count|
+------------+-------------+--------------------+------------+----------+
|          84|Staten Island|Eltingville/Annad...|   Boro Zone|         1|
|         105|    Manhattan|Governor's Island...| Yellow Zone|         1|
|           5|Staten Island|       Arden Heights|   Boro Zone|         1|
|         187|Staten Island|       Port Richmond|   Boro Zone|         3|
|         109|Staten Island|         Great Kills|   Boro Zone|         4|
|         204|Staten Island|   Rossville/Woodrow|   Boro Zone|         4|
|         111|     Brooklyn| Green-Wood Cemetery|   Boro Zone|         4|
|         199|        Bronx|       Rikers Island|   Boro Zone|         4|
|           2|       Queens|         Jamaica Bay|   Boro Zone|         5|
|         251|Staten Island|         Westerleigh|   Boro Zone|        12|
+------------+-------------+----------